In [18]:
import pandas as pd

## Скачай и прочитай CSV-файл, указав ID в качестве индексного столбца.

In [19]:
df = pd.read_csv('../data/auto.csv', sep=',', header=0, index_col='ID')
df.head()

,CarNumber,Make_n_model,Refund,Fines,History
ID,,,,,
0,Y163O8161RUS,Ford Focus,2.0,3200.0,NaN
1,E432XX77RUS,Toyota Camry,1.0,6500.0,NaN
2,7184TT36RUS,Ford Focus,1.0,2100.0,NaN
3,X582HE161RUS,Ford Focus,2.0,2000.0,NaN
4,E34877152RUS,Ford Focus,2.0,6100.0,NaN


## Подсчитай число наблюдений с помощью метода count().

In [20]:
df.count()

CarNumber       931
Make_n_model    931
Refund          914
Fines           869
History          82
dtype: int64

## Удали дубликаты, учитывая только столбцы CarNumber, Make_n_Model и Fines.
- Из двух одинаковых наблюдений оставляй последнее.
- Ещё раз проверь количество наблюдений.

In [21]:
df = df.drop_duplicates(subset=['CarNumber', 'Make_n_model', 'Fines'], keep='last')
df.count()

CarNumber       725
Make_n_model    725
Refund          713
Fines           665
History          65
dtype: int64

## Поработай с пропущенными значениями.
- Посмотри, сколько пропусков в каждом столбце.
- Удали все столбцы с более чем 500 пропусками, используя аргумент thresh. Проверь число пропусков в каждом столбце.
- Замени все пропуски в столбце Refund предыдущим значением в этом столбце (используй аргумент method) и проверь число пропусков.
- Замени все пропуски в столбце Fines средним значением этого столбца (не учитывая NA/NULL при вычислении среднего). Снова проверь число пропусков.

In [22]:
df.isnull().sum()

CarNumber         0
Make_n_model      0
Refund           12
Fines            60
History         660
dtype: int64

In [23]:
df = df.dropna(thresh=len(df)-500, axis=1)
df.isnull().sum()

CarNumber        0
Make_n_model     0
Refund          12
Fines           60
dtype: int64

Из-за несовпадения версий библиотек способ, предлагаемый в задании, кидает ошибку типа:

TypeError                                 Traceback (most recent call last)

Cell In[14], line 1

----> 1 df['Refund'].fillna(method='ffill', inplace=True)

      2 df.isnull().sum()

TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

In [24]:
df['Refund'] = df['Refund'].ffill()
df.isnull().sum()

CarNumber        0
Make_n_model     0
Refund           0
Fines           60
dtype: int64

In [25]:
df['Fines'] = df['Fines'].fillna(df['Fines'].mean())
df.isnull().sum()

CarNumber       0
Make_n_model    0
Refund          0
Fines           0
dtype: int64

## Раздели и распарьсь марку и модель.
- Используй apply и для разбиения, и для извлечения значений в новые столбцы Make и Model.
- Удали столбец Make_n_Model.
- Сохрани датафрейм в файл auto.json в формате ниже:

  [{"CarNumber":"Y163O8161RUS","Refund":2.0,"Fines":3200.0,"Make":"Ford",
"Model":"Focus"},
{"CarNumber":"E432XX77RUS","Refund":1.0,"Fines":6500.0,"Make":"Toyota",
"Model":"Camry"}]

Прежде чем разделить данные, убедимся, что можно Make_n_model на 2 столбца по пробелу

In [26]:
df['Make_n_model'].unique()

<StringArray>
[        'Ford Focus',       'Toyota Camry',      'Skoda Octavia',
  'Volkswagen Passat',    'Volkswagen Golf',         'Volkswagen',
   'Volkswagen Jetta', 'Volkswagen Touareg',     'Toyota Corolla',
               'Audi',        'Ford Mondeo',              'Volvo',
                'BMW']
Length: 13, dtype: str

Поскольку есть автомобили, у которых указана только марка, при извлечении модели машины нужно учитывать возможность пропуска

In [27]:
df['Make'] = df['Make_n_model'].apply(lambda x: x.split()[0])
df['Model'] = df['Make_n_model'].apply(lambda x: x.split()[1] if len(x.split()) > 1 else None)
df.head(1)

,CarNumber,Make_n_model,Refund,Fines,Make,Model
ID,,,,,,
0,Y163O8161RUS,Ford Focus,2.0,3200.0,Ford,Focus


In [28]:
df.drop(columns='Make_n_model', inplace=True)
df.head(1)

,CarNumber,Refund,Fines,Make,Model
ID,,,,,
0,Y163O8161RUS,2.0,3200.0,Ford,Focus


In [29]:
df.to_json('../data/auto.json', orient='records')

In [30]:
df2 = pd.read_json('../data/auto.json', orient='records')

In [31]:
df2.head()

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.0,Ford,Focus
1,E432XX77RUS,1,6500.0,Toyota,Camry
2,7184TT36RUS,1,2100.0,Ford,Focus
3,X582HE161RUS,2,2000.0,Ford,Focus
4,92918M178RUS,1,5700.0,Ford,Focus


In [32]:
df2.count()

CarNumber    725
Refund       725
Fines        725
Make         725
Model        716
dtype: int64

In [33]:
df2['Fines'].mean()

8594.586466165412

In [34]:
df2['Refund'].mean()

1.5172413793103448